# BulkFormer DX Demo RNA Validation (37M)

This notebook summarizes the demo-only RNA validation pass for the `37M` BulkFormer checkpoint.

It loads the generated artifacts from `runs/` and `reports/` rather than re-running the full pipeline inline.

## Scope
- Demo RNA preprocess from `data/demo_count_data.csv`
- Anomaly score and cohort calibration with `37M`
- Synthetic spike-in validation
- Visual QC outputs from `reports/figures/`

## Environment note
- On this macOS setup, `torch-sparse` is unusable at runtime, so the loader falls back to `edge_index + edge_weight` for graph convolution.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path('..').resolve()
REPORTS = ROOT / 'reports'
RUNS = ROOT / 'runs'
FIGURES = REPORTS / 'figures'

run_manifest = json.loads((REPORTS / 'run_manifest.json').read_text())
preprocess_qc = json.loads((REPORTS / 'preprocess_qc_summary.json').read_text())
anomaly_qc = json.loads((REPORTS / 'anomaly_qc_summary.json').read_text())
spike_recovery = pd.read_csv(REPORTS / 'spike_recovery.tsv', sep='\t')
preprocess_report = json.loads((RUNS / 'demo_preprocess_37M' / 'preprocess_report.json').read_text())
cohort_scores = pd.read_csv(RUNS / 'demo_anomaly_score_37M' / 'cohort_scores.tsv', sep='\t')
calibration_summary = pd.read_csv(RUNS / 'demo_anomaly_calibrated_37M' / 'calibration_summary.tsv', sep='\t')


In [ ]:
display(Markdown('## Run Manifest'))
display(pd.json_normalize(run_manifest, sep='.').T.rename(columns={0: 'value'}).head(30))
display(Markdown('## Preprocess Summary'))
display(pd.DataFrame([preprocess_report]).T.rename(columns={0: 'value'}))
display(Markdown('## Preprocess QC Summary'))
display(pd.DataFrame([preprocess_qc]))
display(Markdown('## Anomaly QC Summary'))
display(pd.DataFrame([anomaly_qc]))


In [ ]:
display(Markdown('## Core Cohort Tables'))
display(cohort_scores.head())
display(calibration_summary.head())
display(Markdown('## Spike Recovery Preview'))
display(spike_recovery.sort_values(['sample_id', 'spike_rank']).head(20))


In [ ]:
display(Markdown('## QC Figures'))
for figure_name in [
    'preprocess_tpm_total_hist.png',
    'preprocess_log1p_hist.png',
    'preprocess_valid_gene_fraction.png',
    'preprocess_distribution_compare.png',
    'preprocess_gene_median_compare.png',
    'anomaly_mean_abs_residual_hist.png',
    'anomaly_gene_coverage_hist.png',
    'anomaly_top50_examples.png',
    'anomaly_gene_qc_distributions.png',
    'calibration_empirical_p_hist.png',
    'calibration_by_q_hist.png',
    'calibration_absolute_zscore_hist.png',
    'calibration_absolute_by_p_hist.png',
    'calibration_empirical_significant_count_hist.png',
    'calibration_absolute_significant_count_hist.png',
    'spike_rank_shift.png',
    'spike_rank_improvement_hist.png',
]:
    display(Markdown(f'### {figure_name}'))
    display(Image(filename=str(FIGURES / figure_name)))


## Interpretation Notes

- Preprocessing looks internally consistent: TPM totals are effectively `1e6` for every sample and all `20010` BulkFormer genes are covered.
- `demo_normalized_data.csv` is not the same sample cohort as `demo_count_data.csv` (`967` rows vs `100`), so the comparison is distributional rather than one-to-one.
- The anomaly ranking path responds strongly to synthetic spikes: the spiked genes move sharply upward in rank, and many become significant in the calibrated tables.
- The empirical BY path is conservative (`0` significant genes on average), while the normalized absolute-outlier path is much more permissive and should be interpreted cautiously.
